# RG3 실제분류기3개 후보: RF2＋ET1
실제재학습된같은구간별예측을연결. 사후개발후보/안쪽선택성능/회귀전처리개수를구분한다.


In [1]:
from pathlib import Path
import json,pandas as pd,numpy as np
from sklearn.metrics import confusion_matrix
ROOT=Path.cwd();t=ROOT/"reports/tables"
a=pd.read_csv(t/"rg3_187_small_classifier_predictions.csv",parse_dates=["TimeStamp"])
a=a[a.policy.eq("reference_RF2_original")].copy()
b=pd.read_csv(t/"rg3_188_consolidated_mode_predictions.csv",parse_dates=["TimeStamp"])
b=b[b.policy.eq("ET_thermal_mode6")]
p=a.merge(b[["TimeStamp","heldout_block","actual","prediction"]].rename(columns={"prediction":"ET_single"}),on=["TimeStamp","heldout_block","actual"],validate="one_to_one")
assert len(p)==583 and p.actual.sum()==17
p=p.rename(columns={"prediction":"RF2_base"})
p["RF2_plus_ET1"]=(p.RF2_base.astype(bool)|p.ET_single.astype(bool)).astype(int)
def stats(yy,pr):
 tn,fp,fn,tp=confusion_matrix(yy,pr,labels=[0,1]).ravel()
 return {"TP":int(tp),"FP":int(fp),"FN":int(fn),"TN":int(tn),"F1":float(2*tp/(2*tp+fp+fn)) if tp else 0}
rows=[{"policy":"RF2_base","classifiers":2,**stats(p.actual,p.RF2_base)},
 {"policy":"RF2_plus_ET1_fixed_development","classifiers":3,**stats(p.actual,p.RF2_plus_ET1)}]
old=pd.read_csv(t/"rg3_pressure_seed_majority_predictions.csv",parse_dates=["TimeStamp"])
for (scheme,anchor),part in old.groupby(["scheme","anchor"]):
 if scheme!="leave_one_block_out":continue
 met=stats(part.actual,part.prediction)
 if met["TP"]==8 and met["FP"]==6:
  assert len(part)==583
  rows.insert(0,{"policy":"historical50_development","classifiers":50,**met})
  break
assert rows[0]["classifiers"]==50
d=pd.read_csv(t/"original_labeled_shot_features.csv",parse_dates=["TimeStamp"])
d=d[d["product"].eq("RG3")]
p=p.merge(d[["TimeStamp","Reason"]],on="TimeStamp",validate="one_to_one")
p.to_csv(t/"rg3_189_rf2_plus_et1_predictions.csv",index=False)
p[p.actual.eq(1)].to_csv(t/"rg3_189_rf2_plus_et1_all17_gas.csv",index=False)
p[p.actual.eq(0)&p.RF2_plus_ET1.eq(1)].to_csv(t/"rg3_189_rf2_plus_et1_false_positives.csv",index=False)
pd.DataFrame(rows).to_csv(t/"rg3_189_count_reduction_comparison.csv",index=False)
assert stats(p.actual,p.RF2_plus_ET1)=={"TP":5,"FP":6,"FN":12,"TN":560,"F1":10/28}
for file in ["rg3_187_small_classifier_fit_audit.csv","rg3_188_consolidated_fit_audit.csv"]:
 audit=pd.read_csv(t/file)
 assert audit.timestamp_overlap.eq(0).all() and audit.shared_comparison_block.eq(0).all()
result={"comparison":rows,"true_classifier_count":3,
 "classifiers":["RF std4 seed42","RF residual4 seed42","ET conditional thermal6 multiclass normal/gasA/gasB seed42"],
 "preprocessing_models":"4 normal Ridge regressions for residuals plus normal-neighbor reference; not standalone defect classifiers",
 "KMeans":"used only for supervised gas-mode training labels, not deployed as an extra detector",
 "caveat":"fixed combination selected after evaluation results; not nested selected independent performance",
 "old_model_unchanged":True,"new_full_fit_saved_model_exported":False}
(t/"rg3_189_reduction_check.json").write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding="utf-8")
print(pd.DataFrame(rows).to_string(index=False))
print("\nFOUND GAS\n",p[p.actual.eq(1)&p.RF2_plus_ET1.eq(1)][["TimeStamp","RF2_base","ET_single"]].to_string(index=False))
lines=["# RG3 분류기50개 → 2/3개 축소 비교","",
 "| 분류기 수 | 찾은 가스 | 정상 오탐 | 놓친 가스 | 정상 판정 | F1 |",
 "|---:|---:|---:|---:|---:|---:|",
 "| 기존50 | 8 | 6 | 9 | 560 | 0.516 |",
 "| RF2 | 4 | 6 | 13 | 560 | 0.296 |",
 "| **RF2＋ET1 = 3** | **5** | **6** | **12** | **560** | **0.357** |","",
 "정상566/가스17=583샷, 기존같은5구간으로비교했다. 원본RF2예측583개를정확히재현하고새단일ET를실제로재학습했다.",
 "첫축소187(단일RF/ET/LR)과188(다중클래스통합)을실행했다. 단일열ET가RF2에서놓친가스1개를추가탐지했다.",
 "최종3개후보는187의RF2기준＋188의ET열6개단일분류기를OR로연결한다.시드반복/3개LR평균/12개모드RF/압력27개모델은없다.",
 "이조합은평가결과를본뒤확인한고정개발후보이다. 학습내부에서자동선택한187은TP0/FP1,188은TP0/FP3이었으므로이를숨기고3개조합을독립주성능이라고부르지않는다.",
 "분류기3개이지전처리까지학습객체3개라는뜻은아니다. 잔차피처에정상Ridge회귀4개가추가로필요하고정상최근접검색도있다. 이회귀를빼고진짜단일분류기수준으로합친RF8은이번검증에서개선되지않았다.",
 "새저장추론모델을내보내거나155의기존50개모델을교체한것은아니다. 현재결과는축소후보의재학습검증비교이며원본모델을보존했다.","",
 "## 역할",
 "1. RF 흔들림4: 배럴2/호퍼/쿠션/배럴5의과거10표준편차.",
 "2. RF 정상관계잔차4: 압력/계량/쿠션/금형관계의예상값대비차이. 평균RPM식은정의불확실성이있어물리원인으로단정하지않는다.",
 "3. ET 열6: 배럴−금형온도차/금형온도차/금형변화/금형과거편차/배럴−금형변화차/압력÷충전시간. 정상/가스A/가스B를한분류기안에서학습한다.",
 "RF2는학습내부최대점수99%기준,ET는학습내부정상97.5%기준및열변화하위5%확인조건을사용한다.","",
 "## 잡은 가스5개",p[p.actual.eq(1)&p.RF2_plus_ET1.eq(1)][["TimeStamp","RF2_base","ET_single"]].to_string(index=False),"",
 "## 한계",
 "50개대비가스3개를덜잡는다. 정상오탐은6개로같아서같은성능으로압축에성공했다고볼수는없다.",
 "같은Run에준하는구간별개발검증,기존인과센서이력피처재사용조건이다. 미관측새Run/미래최종검증성능은아니다.",
 "비라벨경보율은이번새3개후보에서아직확인하지않았다. 모델수가줄었다고일반화/안정성이검증되는것은아니다.",
 "commit/push하지않았다.","",
 "## 코드",
 "187_rg3_gas_max3_classifier_refit.ipynb → 188_rg3_gas_consolidated_mode_max3.ipynb → 189_rg3_rf2_plus_et1_reduction_review.ipynb.",
 "RF2기준은전처리회귀를포함하는기존22번을재사용한다. 분류기수와전체반복학습횟수는구분한다."]
(ROOT/"reports/rg3_small_model_reduction_summary.md").write_text("\n".join(lines),encoding="utf-8")


                        policy  classifiers  TP  FP  FN  TN       F1
      historical50_development           50   8   6   9 560 0.516129
                      RF2_base            2   4   6  13 560 0.296296
RF2_plus_ET1_fixed_development            3   5   6  12 560 0.357143

FOUND GAS
           TimeStamp  RF2_base  ET_single
2020-10-22 01:04:15         0          1
2020-10-22 04:46:36         1          0
2020-10-22 05:10:17         1          0
2020-10-23 05:15:46         1          0
2020-10-23 07:15:11         1          0


1574